# **Address Data Quality Problems for Chicago Food Inspection Dataset**

This notebook focuses on identifying and resolving data quality issues in a food inspection dataset. The objective is to preprocess the data to analyze its characteristics in preparation for automatically assigning a quality grade (A-E, where A denotes the highest quality and E the lowest) to each entity based on multiple food-related factors.

## Load Dataset

In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("Food_Inspections_20240215.csv")

# Preview data
df.head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location
0,2589375,PERFECT BEGINNINGS CHILD DEVELOPMENT CENTER,PERFECT BEGINNINGS CHILD DEVELOPMENT CENTER,2385784.0,Daycare Above and Under 2 Years,Risk 1 (High),1500 W 119TH ST,CHICAGO,IL,60643.0,02/08/2024,License,Pass,51. PLUMBING INSTALLED; PROPER BACKFLOW DEVICE...,41.677685,-87.659020,"(41.677685065833224, -87.65901954921286)"
1,2589377,BIG BOSS SPICY FRIED CHICKEN,BIG BOSS SPICY FRIED CHICKEN,2807954.0,Restaurant,Risk 1 (High),2520 S HALSTED ST,CHICAGO,IL,60608.0,02/08/2024,Canvass,Fail,23. PROPER DATE MARKING AND DISPOSITION - Comm...,41.846381,-87.646613,"(41.84638121795965, -87.64661301435865)"
2,2589348,Drummond,Drummond,23021.0,School,Risk 1 (High),1845 W Cortland ST,CHICAGO,IL,60622.0,02/08/2024,Canvass Re-Inspection,Fail,59. PREVIOUS PRIORITY FOUNDATION VIOLATION COR...,41.915886,-87.674496,"(41.91588633998172, -87.67449563016454)"
3,2589332,SUBWAY,SUBWAY,2665377.0,Restaurant,Risk 1 (High),1608 W 59TH ST,CHICAGO,IL,60636.0,02/08/2024,Canvass Re-Inspection,Pass,57. ALL FOOD EMPLOYEES HAVE FOOD HANDLER TRAIN...,41.786850,-87.664801,"(41.786849792502515, -87.66480141216883)"
4,2589295,LUCY'S,LUCY'S,2712942.0,Restaurant,Risk 1 (High),4570 N BROADWAY,CHICAGO,IL,60640.0,02/07/2024,Canvass Re-Inspection,Pass,NaN,41.965270,-87.657554,"(41.96527014438852, -87.6575542920256)"


## Basic Overview

In [2]:
df.info()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 267531 entries, 0 to 267530
Data columns (total 17 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Inspection ID    267531 non-null  int64  
 1   DBA Name         267531 non-null  object 
 2   AKA Name         265058 non-null  object 
 3   License #        267513 non-null  float64
 4   Facility Type    262416 non-null  object 
 5   Risk             267450 non-null  object 
 6   Address          267531 non-null  object 
 7   City             267370 non-null  object 
 8   State            267472 non-null  object 
 9   Zip              267482 non-null  float64
 10  Inspection Date  267531 non-null  object 
 11  Inspection Type  267530 non-null  object 
 12  Results          267531 non-null  object 
 13  Violations       194191 non-null  object 
 14  Latitude         266608 non-null  float64
 15  Longitude        266608 non-null  float64
 16  Location         266608 non-null  obje

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location
count,2.675310e+05,267531,265058,2.675130e+05,262416,267450,267531,267370,267472,267482.000000,267531,267530,267531,194191,266608.000000,266608.000000,266608
unique,NaN,32165,30614,NaN,513,4,19639,78,5,NaN,3562,110,7,192923,NaN,NaN,18210
top,NaN,SUBWAY,SUBWAY,NaN,Restaurant,Risk 1 (High),11601 W TOUHY AVE,CHICAGO,IL,NaN,11/14/2013,Canvass,Pass,32. FOOD AND NON-FOOD CONTACT SURFACES PROPERL...,NaN,NaN,"(42.008536400868735, -87.91442843927047)"
freq,NaN,3552,4370,NaN,179811,196048,3273,266472,267461,NaN,185,139061,137733,11,NaN,NaN,3291
mean,1.729715e+06,NaN,NaN,1.721737e+06,NaN,NaN,NaN,NaN,NaN,60628.704040,NaN,NaN,NaN,NaN,41.880728,-87.676427,NaN
std,7.194335e+05,NaN,NaN,9.269263e+05,NaN,NaN,NaN,NaN,NaN,148.879568,NaN,NaN,NaN,NaN,0.081072,0.058636,NaN
min,4.424700e+04,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,NaN,10014.000000,NaN,NaN,NaN,NaN,41.644670,-87.914428,NaN
25%,1.307352e+06,NaN,NaN,1.336815e+06,NaN,NaN,NaN,NaN,NaN,60614.000000,NaN,NaN,NaN,NaN,41.831819,-87.707462,NaN
50%,1.950830e+06,NaN,NaN,2.048784e+06,NaN,NaN,NaN,NaN,NaN,60625.000000,NaN,NaN,NaN,NaN,41.891770,-87.666419,NaN
75%,2.356946e+06,NaN,NaN,2.369039e+06,NaN,NaN,NaN,NaN,NaN,60643.000000,NaN,NaN,NaN,NaN,41.939787,-87.634942,NaN


## Missing Values

In [3]:
# Count missing values
missing = df.isnull().sum()

# Percentage missing
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values(by='Missing %', ascending=False)

missing_df

,Missing Count,Missing %
Violations,73340,27.413646
Facility Type,5115,1.911928
AKA Name,2473,0.924379
Longitude,923,0.345007
Location,923,0.345007
Latitude,923,0.345007
City,161,0.060180
Risk,81,0.030277
State,59,0.022054
Zip,49,0.018316


## Duplicate Records

Each `Inspection ID` should be unique.

In [4]:
# Check duplicate Inspection IDs
duplicates = df[df.duplicated(subset=['Inspection ID'])]

len(duplicates)

0

## Column Values Verification -> 'Location' is a concatenation of 'Latitude' and 'Longitude'

In [5]:
# Keep only rows with complete geo info
df = df.dropna(subset=["Latitude", "Longitude", "Location"])

# Ensure numeric
df["Latitude"] = df["Latitude"].astype(float)
df["Longitude"] = df["Longitude"].astype(float)

# Extract numbers from Location (simple approach)
df["Location"] = df["Location"].astype(str)
df[["loc_lat", "loc_lon"]] = df["Location"].str.extract(r"(-?\d+\.\d+)[^\d-]+(-?\d+\.\d+)")

df["loc_lat"] = df["loc_lat"].astype(float)
df["loc_lon"] = df["loc_lon"].astype(float)

# Compare with small tolerance
tolerance = 0.0001

mismatches = df[
    (np.abs(df["Latitude"] - df["loc_lat"]) > tolerance) |
    (np.abs(df["Longitude"] - df["loc_lon"]) > tolerance)
]

print(f"Total rows checked : {len(df):,}")
print(f"Mismatches         : {len(mismatches):,}")

# Drop helper columns
df = df.drop(columns=["loc_lat", "loc_lon"])

Total rows checked : 266,608
Mismatches         : 0


## Inconsistent Formats and Invalid Values

Descriptions of the data elements included in the dataset: [https://data.cityofchicago.org/api/assets/BAD5301B-681A-4202-9D25-51B2CAE672FF](https://data.cityofchicago.org/api/assets/BAD5301B-681A-4202-9D25-51B2CAE672FF)

### Facility Type

Each establishment is described by one of the following: bakery, banquet
hall, candy store, caterer, coffee shop, day care center (for ages less than 2), day care
center (for ages 2 – 6), day care center (combo, for ages less than 2 and 2 – 6
combined), gas station, Golden Diner, grocery store, hospital, long term care
center(nursing home), liquor store, mobile food dispenser, restaurant, paleteria, school,
shelter, tavern, social club, wholesaler, or Wrigley Field Rooftop.

In [7]:
df['Facility Type'].nunique()

508

In [8]:
df['Facility Type'].unique()

array(['Daycare Above and Under 2 Years', 'Restaurant', 'School',
       "Children's Services Facility", 'Grocery Store',
       'Daycare (Under 2 Years)', nan, 'Golden Diner',
       'Pop-Up Establishment Host-Tier III', 'Bakery', 'Catering',
       'Hospital', 'PASTRY SCHOOL', 'PACKAGED HEALTH FOODS', 'Liquor',
       'Daycare (2 - 6 Years)', 'MILK TEA', 'Mobile Food Preparer',
       'Long Term Care', 'LIVE POULTRY', 'PRIVATE SCHOOL',
       'GAS STATION/GROCERY', 'Shared Kitchen',
       'Shared Kitchen User (Long Term)', 'Banquet', 'BANQUET HALL',
       "1023 CHILDERN'S SERVICES FACILITY", "CHILDERN'S SERVICE FACILITY",
       'RESTAURANT/BAR', 'CHILDRENS SERVICES FACILITY', 'CULINARY SCHOOL',
       'Assisted Living', 'Special Event', 'bar', 'GROCERY/BAKERY',
       'ROOFTOPS', 'RIVERWALK', 'CATERING/CAFE', 'ROOFTOP',
       'Banquet Hall', 'BANQUET', 'STADIUM',
       'Mobile Frozen Desserts Vendor', 'COOKING SCHOOL', 'COMMISSARY',
       'ARCHDIOCESE', 'Navy Pier Kiosk', 'JUIC

In [9]:
df['Facility Type'] = (
    df['Facility Type']
    .str.upper()
    .str.strip()
)

df['Facility Type'].nunique()

454

In [10]:
df['Facility Type'].unique()

array(['DAYCARE ABOVE AND UNDER 2 YEARS', 'RESTAURANT', 'SCHOOL',
       "CHILDREN'S SERVICES FACILITY", 'GROCERY STORE',
       'DAYCARE (UNDER 2 YEARS)', nan, 'GOLDEN DINER',
       'POP-UP ESTABLISHMENT HOST-TIER III', 'BAKERY', 'CATERING',
       'HOSPITAL', 'PASTRY SCHOOL', 'PACKAGED HEALTH FOODS', 'LIQUOR',
       'DAYCARE (2 - 6 YEARS)', 'MILK TEA', 'MOBILE FOOD PREPARER',
       'LONG TERM CARE', 'LIVE POULTRY', 'PRIVATE SCHOOL',
       'GAS STATION/GROCERY', 'SHARED KITCHEN',
       'SHARED KITCHEN USER (LONG TERM)', 'BANQUET', 'BANQUET HALL',
       "1023 CHILDERN'S SERVICES FACILITY", "CHILDERN'S SERVICE FACILITY",
       'RESTAURANT/BAR', 'CHILDRENS SERVICES FACILITY', 'CULINARY SCHOOL',
       'ASSISTED LIVING', 'SPECIAL EVENT', 'BAR', 'GROCERY/BAKERY',
       'ROOFTOPS', 'RIVERWALK', 'CATERING/CAFE', 'ROOFTOP', 'STADIUM',
       'MOBILE FROZEN DESSERTS VENDOR', 'COOKING SCHOOL', 'COMMISSARY',
       'ARCHDIOCESE', 'NAVY PIER KIOSK', 'JUICE BAR/GROCERY',
       'POP-UP EST

### Risk

In [11]:
df['Risk'].unique()

array(['Risk 1 (High)', 'Risk 3 (Low)', 'Risk 2 (Medium)', 'All', nan],
      dtype=object)

### City

In [12]:
df['City'].unique()

array(['CHICAGO', 'Chicago', nan, 'CCHICAGO', 'chicago', 'CHICAGOCHICAGO',
       'chicagoBEDFORD PARK', 'CHICAGO.', 'CHicago', 'CHCHICAGO',
       'CHICAGOI', 'WESTMONT', 'LOMBARD', 'SUMMIT', 'INACTIVE', 'alsip',
       'CHARLES A HAYES', 'CHICAGOO', '312CHICAGO', 'CHICAGOC',
       'CHCICAGO', 'BLUE ISLAND'], dtype=object)

In [13]:
df['City'] = (
    df['City']
    .str.upper()
    .str.strip()
)
df['City'] = df['City'].str.replace(r'.*CHICAGO.*', 'CHICAGO', regex=True)

df['City'].unique()

array(['CHICAGO', nan, 'WESTMONT', 'LOMBARD', 'SUMMIT', 'INACTIVE',
       'ALSIP', 'CHARLES A HAYES', 'CHCICAGO', 'BLUE ISLAND'],
      dtype=object)

In [14]:
df['City'] = df['City'].replace({
    'CHCICAGO': 'CHICAGO'
})

# Keep ONLY Chicago rows
df = df[df['City'] == 'CHICAGO']

### State

In [15]:
df['State'].value_counts(dropna=False)

,count
State,
IL,266389
NaN,41


In [16]:
df = df[df['State'] == 'IL']

### Inspection Date

In [17]:
future_cutoff = pd.Timestamp("2024-02-15")

# Convert date
df['Inspection Date'] = pd.to_datetime(df['Inspection Date'], errors='coerce')

# Future dates
future_dates = df[df['Inspection Date'] > future_cutoff]

len(future_dates)

0

### Inspection Type

In [18]:
df['Inspection Type'].value_counts(dropna=False)

,count
Inspection Type,
Canvass,138473
License,35451
Canvass Re-Inspection,29325
Complaint,25307
License Re-Inspection,11388
...,...
POSSIBLE FBI,1
TASK FORCE NOT READY,1
TASK FORCE LIQUOR (1481),1


In [19]:
df['Inspection Type'] = (
    df['Inspection Type']
    .str.upper()
    .str.strip()
)

df['Inspection Type'].unique()

array(['LICENSE', 'CANVASS', 'CANVASS RE-INSPECTION', 'COMPLAINT',
       'NON-INSPECTION', 'COMPLAINT RE-INSPECTION',
       'LICENSE RE-INSPECTION', 'SUSPECTED FOOD POISONING RE-INSPECTION',
       'SHORT FORM COMPLAINT', 'SUSPECTED FOOD POISONING',
       'RECENT INSPECTION', 'NOT READY', 'OUT OF BUSINESS',
       'CONSULTATION', nan, 'KITCHEN CLOSED FOR RENOVATION',
       'SHORT FORM FIRE-COMPLAINT', 'O.B.', 'SPECIAL EVENTS (FESTIVALS)',
       'CORRECTIVE ACTION', 'OWNER SUSPENDED OPERATION/LICENSE',
       'LICENSE CONSULTATION', 'TASK FORCE LIQUOR 1475', 'COMPLAINT-FIRE',
       'TAG REMOVAL', 'FIRE COMPLAINT', 'LICENSE-TASK FORCE',
       'PRE-LICENSE CONSULTATION', 'CANVASS SCHOOL/SPECIAL EVENT',
       'COMPLAINT-FIRE RE-INSPECTION', 'NO ENTRY', 'PACKAGE LIQUOR 1474',
       'TASK FORCE LIQUOR 1470', 'TASK FORCE FOR LIQUOR 1474', 'ADDENDUM',
       'TASK FORCE LIQUOR INSPECTION 1474', 'SFP/COMPLAINT',
       'TASK FORCE NIGHT', 'SFP', 'LICENSE REQUEST', 'ILLEGAL OPERATION',


In [20]:
df['Inspection Type'] = df['Inspection Type'].replace({
    'CANVAS': 'CANVASS',
    'CANVASS SCHOOL/SPECIAL EVENT': 'CANVASS SPECIAL EVENTS',
    'CANVASS/SPECIAL EVENT': 'CANVASS SPECIAL EVENTS',
    'FIRE/COMPLAIN': 'FIRE COMPLAINT',
    "KIDS CAFE'": 'KIDS CAFE',
    'LICENSE TASK FORCE / NOT -FOR-PROFIT CLU': 'LICENSE TASK FORCE / NOT -FOR-PROFIT CLUB',
    'O.B.': 'OUT OF BUSINESS',
    'OUT OFBUSINESS' :'OUT OF BUSINESS',
    'TASK FORCE FOR LIQUOR 1474': 'TASK FORCE LIQUOR 1474',

})

df['Inspection Type'].value_counts(dropna=False)

,count
Inspection Type,
CANVASS,138475
LICENSE,35453
CANVASS RE-INSPECTION,29325
COMPLAINT,25307
LICENSE RE-INSPECTION,11388
...,...
POSSIBLE FBI,1
TASK FORCE NOT READY,1
TASK FORCE LIQUOR (1481),1


### Results

In [21]:
df['Results'].unique()

array(['Pass', 'Fail', 'Pass w/ Conditions', 'No Entry', 'Not Ready',
       'Out of Business', 'Business Not Located'], dtype=object)

## Save into CSV File

In [22]:
df.to_csv('Preprocessed_Food_Inspections_20240215.csv', index=False)